# Events PDF → parquet corregido y enriquecido

Este notebook:

1. Extrae eventos desde los PDFs.
2. Conserva una fila por evento con `cow_id + date + event_type`.
3. Enriquece `raw_event_text` con variables estructuradas.
4. Genera un dataset diario por `cow_id + date` con:
   - `event_count`
   - one-hot encoding de `event_type`
   - banderas clínicas
   - campos agregados útiles para ML

La llave analítica final del proyecto debe seguir siendo:

`cow_id + date`

In [1]:
from pathlib import Path
import re
import unicodedata
import numpy as np
import pandas as pd
import pdfplumber

In [2]:
# Ajusta estas rutas a tu proyecto
PDF_DIR = Path("../data/raw/eventos")
OUTPUT_EVENTS = Path("../data/interim/events.parquet")
OUTPUT_EVENTS_DAILY = Path("../data/interim/events_daily.parquet")

pdf_files = sorted(PDF_DIR.glob("*.pdf"))

print("PDF encontrados:", len(pdf_files))
for f in pdf_files[:10]:
    print("-", f.name)
if len(pdf_files) > 10:
    print("...")

PDF encontrados: 69
- Eventos de animales 1204.pdf
- Eventos de animales 1213.pdf
- Eventos de animales 1216.pdf
- Eventos de animales 1225.pdf
- Eventos de animales 1228.pdf
- Eventos de animales 1235.pdf
- Eventos de animales 1236.pdf
- Eventos de animales 1242.pdf
- Eventos de animales 1243.pdf
- Eventos de animales 1497.pdf
...


In [3]:
EVENT_KEYWORDS = [
    "Cambio ID Tran",
    "Cambio ID",
    "Cambio de grup",
    "Cambio tabla ali",
    "Condición corpo",
    "Secado",
    "Diagnósticos/Tr",
    "Control de Gest",
    "Invitación Visita",
    "Inseminación",
    "Celo",
    "Parto",
    "Peso",
    "Entrada",
]

EVENT_MAP = {
    "Cambio ID": "Cambio_ID",
    "Cambio ID Tran": "Cambio_ID_Tran",
    "Cambio de grup": "Cambio_Grupo",
    "Cambio tabla ali": "Cambio_Tabla_Alim",
    "Celo": "Celo",
    "Condición corpo": "Condicion_Corporal",
    "Control de Gest": "Control_Gestacion",
    "Diagnósticos/Tr": "Diagnostico_Tratamiento",
    "Entrada": "Entrada",
    "Inseminación": "Inseminacion",
    "Invitación Visita": "Invitacion_Visita",
    "Parto": "Parto",
    "Peso": "Peso",
    "Secado": "Secado",
}

HEADER_PATTERNS = [
    re.compile(r"^Evento\s+Tipo de evento", re.IGNORECASE),
    re.compile(r"^Fecha$", re.IGNORECASE),
    re.compile(r"^evento$", re.IGNORECASE),
    re.compile(r"^del$", re.IGNORECASE),
    re.compile(r"^Descripción\s+Usuario\s+Comentario$", re.IGNORECASE),
]

def NormalizeLine(text):
    return re.sub(r"\s+", " ", str(text)).strip()

def NormalizeText(text):
    text = "" if pd.isna(text) else str(text)
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def IsHeaderLine(line):
    return any(p.search(line) for p in HEADER_PATTERNS)

def ExtractAnimalIdFromFilename(pdf_path):
    match = re.search(r"(\d+)", pdf_path.stem)
    return int(match.group(1)) if match else None

def ExtractLactationMarker(line):
    match = re.search(r"N[ºo]\s*Lactaci[oó]n\s*(\d+)", line, re.IGNORECASE)
    return int(match.group(1)) if match else None

def LooksLikeEventStart(line):
    for kw in EVENT_KEYWORDS:
        if re.match(rf"^{re.escape(kw)}\s+\d{{2}}/\d{{2}}/\d{{1,4}}\b", line):
            return True
    return False

def SplitEventType(line):
    for kw in sorted(EVENT_KEYWORDS, key=len, reverse=True):
        if re.match(rf"^{re.escape(kw)}\s+\d{{2}}/\d{{2}}/\d{{1,4}}\b", line):
            return kw
    return None

def ExtractFullYearAnchor(text):
    years = [int(y) for y in re.findall(r"\b\d{2}/\d{2}/(20\d{2})\b", str(text))]
    return years[0] if years else np.nan

def MonthDayFromToken(date_token):
    if not isinstance(date_token, str):
        return (None, None)

    m = re.match(r"(\d{2})/(\d{2})/(\d{1,4})", date_token)
    if not m:
        return (None, None)

    day = int(m.group(1))
    month = int(m.group(2))
    return (month, day)

def SafeBoolContains(text, patterns):
    t = NormalizeText(text).lower()
    if isinstance(patterns, str):
        patterns = [patterns]
    return any(re.search(p, t) for p in patterns)

def ExtractField(text, key):
    text = "" if pd.isna(text) else str(text)
    pattern = rf"{re.escape(key)}\s*:\s*(.*?)(?=(?:\b(?:Dns|Med|Trat|Loc\.)\s*:)|$)"
    m = re.search(pattern, text, re.IGNORECASE)
    return m.group(1).strip(" ;,-") if m else None

def ExtractFloatAfterPattern(text, pattern):
    text = "" if pd.isna(text) else str(text)
    m = re.search(pattern, text, re.IGNORECASE)
    return float(m.group(1)) if m else np.nan

def ExtractIntAfterPattern(text, pattern):
    text = "" if pd.isna(text) else str(text)
    m = re.search(pattern, text, re.IGNORECASE)
    return int(m.group(1)) if m else np.nan

In [4]:
def ParsePdfEvents(pdf_path):
    cow_id = ExtractAnimalIdFromFilename(pdf_path)
    rows = []

    with pdfplumber.open(pdf_path) as pdf:
        current_lactation = np.nan
        current_block = 0
        current_event = None
        event_order = 0

        def flush_current():
            nonlocal current_event, event_order

            if current_event is None:
                return

            current_event["event_order"] = event_order
            current_event["raw_event_text"] = " ".join(
                [x for x in current_event.pop("parts") if NormalizeLine(x)]
            ).strip()

            rows.append(current_event)
            event_order += 1
            current_event = None

        for page_idx, page in enumerate(pdf.pages, start=1):
            text = page.extract_text() or ""
            lines = [NormalizeLine(x) for x in text.splitlines() if NormalizeLine(x)]

            for line in lines:
                if IsHeaderLine(line):
                    continue

                lact = ExtractLactationMarker(line)
                if lact is not None:
                    flush_current()
                    current_lactation = lact
                    current_block += 1
                    continue

                if LooksLikeEventStart(line):
                    flush_current()

                    event_type = SplitEventType(line)
                    tail = line[len(event_type):].strip()

                    m = re.match(r"(?P<date_token>\d{2}/\d{2}/\d{1,4})\s*(?P<rest>.*)", tail)

                    current_event = {
                        "cow_id": cow_id,
                        "source_file": pdf_path.name,
                        "page_number": page_idx,
                        "block_id": current_block,
                        "lactation_number": current_lactation,
                        "event_type": event_type,
                        "event_type_clean": EVENT_MAP.get(event_type),
                        "date_token": m.group("date_token") if m else None,
                        "parts": [m.group("rest").strip()] if m and m.group("rest").strip() else [],
                    }
                else:
                    if current_event is not None:
                        current_event["parts"].append(line)

        flush_current()

    return pd.DataFrame(rows)

def InferYearsWithinBlock(df_block):
    df_block = df_block.copy().reset_index(drop=True)
    df_block["anchor_year"] = df_block["raw_event_text"].apply(ExtractFullYearAnchor)

    years = [None] * len(df_block)
    anchor_idx = [i for i, y in enumerate(df_block["anchor_year"]) if pd.notna(y)]

    if not len(df_block):
        return df_block

    if not anchor_idx:
        years[0] = 2025
        start_idx = 0
    else:
        for i in anchor_idx:
            years[i] = int(df_block.loc[i, "anchor_year"])

        first_anchor = anchor_idx[0]
        for i in range(first_anchor - 1, -1, -1):
            prev_month, prev_day = MonthDayFromToken(df_block.loc[i, "date_token"])
            cur_month, cur_day = MonthDayFromToken(df_block.loc[i + 1, "date_token"])
            years[i] = years[i + 1] + (1 if (prev_month, prev_day) < (cur_month, cur_day) else 0)

        for a, b in zip(anchor_idx, anchor_idx[1:]):
            for i in range(a + 1, b):
                this_month, this_day = MonthDayFromToken(df_block.loc[i, "date_token"])
                prev_month, prev_day = MonthDayFromToken(df_block.loc[i - 1, "date_token"])
                years[i] = years[i - 1] - (1 if (this_month, this_day) > (prev_month, prev_day) else 0)

        start_idx = anchor_idx[-1]

    for i in range(start_idx + 1, len(df_block)):
        this_month, this_day = MonthDayFromToken(df_block.loc[i, "date_token"])
        prev_month, prev_day = MonthDayFromToken(df_block.loc[i - 1, "date_token"])
        years[i] = years[i - 1] - (1 if (this_month, this_day) > (prev_month, prev_day) else 0)

    for i in range(1, len(df_block)):
        if years[i] is None:
            this_month, this_day = MonthDayFromToken(df_block.loc[i, "date_token"])
            prev_month, prev_day = MonthDayFromToken(df_block.loc[i - 1, "date_token"])
            years[i] = years[i - 1] - (1 if (this_month, this_day) > (prev_month, prev_day) else 0)

    df_block["year_inferred"] = years
    df_block["date_str"] = df_block.apply(
        lambda r: (
            f"{int(r['date_token'][:2]):02d}/{int(r['date_token'][3:5]):02d}/{int(r['year_inferred'])}"
            if isinstance(r["date_token"], str) and pd.notna(r["year_inferred"])
            else None
        ),
        axis=1,
    )
    df_block["date"] = pd.to_datetime(df_block["date_str"], dayfirst=True, errors="coerce")
    return df_block

In [5]:
all_events = []

for pdf_file in pdf_files:
    try:
        df_pdf = ParsePdfEvents(pdf_file)

        if df_pdf.empty:
            continue

        df_pdf = (
            df_pdf
            .sort_values(["block_id", "event_order"], ascending=[True, True])
            .reset_index(drop=True)
        )

        df_pdf = (
            df_pdf
            .groupby(["source_file", "block_id"], group_keys=False)
            .apply(InferYearsWithinBlock)
            .reset_index(drop=True)
        )

        all_events.append(df_pdf)

    except Exception as e:
        print(f"Error en {pdf_file.name}: {e}")

events_df = pd.concat(all_events, ignore_index=True) if all_events else pd.DataFrame()

print("Eventos extraídos:", events_df.shape)
events_df.head(20)

Eventos extraídos: (6685, 12)


,cow_id,page_number,lactation_number,event_type,event_type_clean,date_token,event_order,raw_event_text,anchor_year,year_inferred,date_str,date
0,1204,1,1,Cambio tabla ali,Cambio_Tabla_Alim,22/08/2,0,User1 Condición corporal: 3.25 - DDUP:,NaN,2025,22/08/2025,2025-08-22
1,1204,1,1,Condición corpo,Condicion_Corporal,22/08/2,1,"473 User1 Dry Off SE SECO CON 473 DEL, 20.75 L...",NaN,2025,22/08/2025,2025-08-22
2,1204,1,1,Secado,Secado,22/08/2,2,User1 GESTANTE EN LA 7 INSEM. SE SECO PORQUE T...,NaN,2025,22/08/2025,2025-08-22
3,1204,1,1,Cambio de grup,Cambio_Grupo,22/08/2,3,User1 Dns: VACUNA; Loc.: DD; Trat:,NaN,2025,22/08/2025,2025-08-22
4,1204,1,1,Diagnósticos/Tr,Diagnostico_Tratamiento,22/08/2,4,starvacc; Med: STARVAC 1 d 1X2ml User1 Intramu...,NaN,2025,22/08/2025,2025-08-22
5,1204,1,1,Diagnósticos/Tr,Diagnostico_Tratamiento,22/08/2,5,ROTAVEC; Med: User1 Dns: Secado por baja; Loc....,NaN,2025,22/08/2025,2025-08-22
6,1204,1,1,Diagnósticos/Tr,Diagnostico_Tratamiento,22/08/2,6,Trat: CEFA SAFE; Med: CEFA SAFE 1 User1 d 1X4p...,NaN,2025,22/08/2025,2025-08-22
7,1204,1,1,Condición corpo,Condicion_Corporal,22/08/2,7,473 Admin + CARGADA DE 124,NaN,2025,22/08/2025,2025-08-22
8,1204,1,1,Control de Gest,Control_Gestacion,21/08/2,8,User1 + CARGADA DE 96,NaN,2025,21/08/2025,2025-08-21
9,1204,1,1,Control de Gest,Control_Gestacion,24/07/2,9,"User1 24/07/2025, Reconfirmacion 1 (90-",2025.0,2025,24/07/2025,2025-07-24


In [6]:

# Enriquecimiento semántico de raw_event_text + normalización clínica

NOISE_EXACT = {
    "",
    "user1",
    "user1 +",
    "delproclien",
    "unknown brand",
    "unknownbrand",
    "mvz luis z",
}

def CleanExtractedText(value):
    if pd.isna(value):
        return np.nan
    value = NormalizeLine(value)
    if not value:
        return np.nan

    norm = NormalizeText(value).lower()

    # ruido común
    if norm in NOISE_EXACT:
        return np.nan
    if norm in {"-", "--", "---", "+", "na", "n/a"}:
        return np.nan

    # textos claramente administrativos sin contenido clínico
    if re.fullmatch(r"user\d+", norm):
        return np.nan

    return value

def NormalizeClinicalText(value):
    if pd.isna(value):
        return np.nan
    value = NormalizeText(value).lower()
    value = re.sub(r"\s+", " ", value).strip()
    if not value:
        return np.nan
    return value

def MapDiagnosisCategory(value):
    x = NormalizeClinicalText(value)
    if pd.isna(x):
        return np.nan

    if "mastit" in x:
        return "diag_mastitis"
    if "vacuna" in x:
        return "diag_vacuna"
    if "vacia" in x:
        return "diag_vacia"
    if "diarrea" in x or "digest" in x:
        return "diag_digestivo"
    if "podal" in x or "cojera" in x or "lameness" in x:
        return "diag_problema_podal"
    if "revision" in x:
        return "diag_revision"
    if x == "alta" or " alta " in f" {x} ":
        return "diag_alta"
    if "secado por baja" in x:
        return "diag_secado_baja"
    if x == "secar" or "secado" in x:
        return "diag_secado"
    if "presynch" in x:
        return "diag_presynch"
    if "protocolo sincronizacion" in x or "sincronizacion" in x:
        return "diag_sincronizacion"
    if "vitamina" in x:
        return "diag_vitamina"
    if "bolo rumensin" in x or ("rumensin" in x and "bolo" in x):
        return "diag_bolo_rumensin"
    if "gnrh" in x:
        return "diag_gnrh"
    if re.fullmatch(r"pg\d*", x) or "pg " in x:
        return "diag_pg"

    return "diag_other"

def MapTreatmentCategory(value):
    x = NormalizeClinicalText(value)
    if pd.isna(x):
        return np.nan

    if "pg2" in x or re.fullmatch(r"pg", x) or "pg1" in x or "pg x1" in x or "pg x2" in x or "lutalyse" in x:
        return "trt_pg"
    if "gnrh" in x:
        return "trt_gnrh"
    if "cidr" in x:
        return "trt_cidr"
    if "cefa" in x:
        return "trt_cefa"
    if "bolo rumensin" in x or ("rumensin" in x and "bolo" in x):
        return "trt_bolo_rumensin"
    if any(k in x for k in ["spectramast", "hidropen", "septotryl", "tylan", "matjet", "rilexine", "one"]):
        return "trt_antibiotico"
    if any(k in x for k in ["bovishield", "starvac"]):
        return "trt_vacuna"
    if "vitamina" in x:
        return "trt_vitamina"

    return "trt_other"

def MapMedicationCategory(value):
    x = NormalizeClinicalText(value)
    if pd.isna(x):
        return np.nan

    if "gnrh" in x:
        return "med_gnrh"
    if "cidr" in x:
        return "med_cidr"
    if "cefa" in x:
        return "med_cefa"
    if "rumensin" in x:
        return "med_bolo_rumensin"
    if any(k in x for k in ["spectramast", "hidropen", "septotryl", "tylan", "matjet", "rilexine"]):
        return "med_antibiotico"
    if any(k in x for k in ["starvac", "bovishield"]):
        return "med_vacuna"
    if "lutalyse" in x or re.fullmatch(r"pg\d*", x) or " pg " in f" {x} ":
        return "med_pg"

    return "med_other"

def EnrichEventRow(row):
    raw = row["raw_event_text"]
    norm = NormalizeText(raw).lower()
    event_type = row["event_type"]

    diagnosis_text = CleanExtractedText(ExtractField(raw, "Dns"))
    medication_text = CleanExtractedText(ExtractField(raw, "Med"))
    treatment_text = CleanExtractedText(ExtractField(raw, "Trat"))
    location_text = CleanExtractedText(ExtractField(raw, "Loc."))

    diagnosis_norm = NormalizeText(diagnosis_text).lower() if pd.notna(diagnosis_text) else ""
    medication_norm = NormalizeText(medication_text).lower() if pd.notna(medication_text) else ""
    treatment_norm = NormalizeText(treatment_text).lower() if pd.notna(treatment_text) else ""
    combined_norm = " | ".join([norm, diagnosis_norm, medication_norm, treatment_norm])

    body_condition_score = np.nan
    if event_type == "Condición corpo":
        body_condition_score = ExtractFloatAfterPattern(raw, r"Condici[oó]n\s+corporal\s*:\s*(\d+(?:\.\d+)?)")
        if pd.isna(body_condition_score):
            body_condition_score = ExtractFloatAfterPattern(raw, r"\b(\d(?:\.\d+)?)\b")

    weight_kg = np.nan
    if event_type == "Peso":
        weight_kg = ExtractFloatAfterPattern(raw, r"Peso\s*:?\s*(\d+(?:\.\d+)?)")
        if pd.isna(weight_kg):
            weight_kg = ExtractFloatAfterPattern(raw, r"\b(\d{3}(?:\.\d+)?)\s*kg\b")

    liters_at_dryoff = np.nan
    dcc_at_dryoff = np.nan
    del_at_dryoff = np.nan
    gestant_insem_number = np.nan
    if event_type == "Secado":
        liters_at_dryoff = ExtractFloatAfterPattern(raw, r"(\d+(?:\.\d+)?)\s*LITROS")
        dcc_at_dryoff = ExtractIntAfterPattern(raw, r"(\d+)\s*DCC")
        del_at_dryoff = ExtractIntAfterPattern(raw, r"(\d+)\s*DEL")
        gestant_insem_number = ExtractIntAfterPattern(raw, r"QUEDO\s+GESTANTE\s+EN\s+LA\s+(\d+)\s+INSEM")

    from_group = None
    to_group = None
    if event_type in {"Cambio de grup", "Cambio tabla ali", "Secado"}:
        m = re.search(r"([^;]+?)\s*->\s*([^;]+)", raw)
        if m:
            from_group = NormalizeLine(m.group(1))
            to_group = NormalizeLine(m.group(2))

    semen_code = None
    if event_type == "Inseminación":
        m = re.search(r"\b(\d{10,16})\b", raw)
        semen_code = m.group(1) if m else None

    diagnosis_category = MapDiagnosisCategory(diagnosis_text)
    treatment_category = MapTreatmentCategory(treatment_text)
    medication_category = MapMedicationCategory(medication_text)

    pregnancy_positive = int(SafeBoolContains(combined_norm, [r"\bgestante\b", r"\bprenad", r"\bpreñad", r"reconfirmacion", r"diagnostico de gestacion"]))
    pregnancy_negative = int(SafeBoolContains(combined_norm, [r"\bvacia\b", r"comprobacion de preñez\(-\)", r"vacia de"]))
    mastitis_flag = int(SafeBoolContains(combined_norm, [r"mastit"]))
    digestive_flag = int(SafeBoolContains(combined_norm, [r"digest", r"diarrea"]))
    vaccine_flag = int(SafeBoolContains(combined_norm, [r"vacun", r"starvac", r"bovilis", r"virashield", r"bovishield"]))
    reproductive_flag = int(SafeBoolContains(combined_norm, [r"insem", r"celo", r"cidr", r"presynch", r"ovsynch", r"gestacion", r"vacia", r"aborto", r"gnrh", r"\bpg\b"]))
    abortion_flag = int(SafeBoolContains(combined_norm, [r"aborto"]))
    antibiotic_flag = int(SafeBoolContains(combined_norm, [r"rilexine", r"hidropen", r"spectramast", r"mastijet", r"cefa", r"cef", r"septotryl", r"tylan", r"matjet", r"\bone\b"]))
    hormone_flag = int(SafeBoolContains(combined_norm, [r"cidr", r"\bpg\b", r"presynch", r"ovsynch", r"gnrh", r"lutalyse"]))
    dryoff_related_flag = int(SafeBoolContains(combined_norm, [r"dry off", r"se seco", r"secar", r"secas", r"secado"]))
    fresh_cow_flag = int(SafeBoolContains(combined_norm, [r"vacas frescas", r"fresca"]))
    rumen_bolus_flag = int(SafeBoolContains(combined_norm, [r"rumensin", r"bolo"]))
    heat_flag = int(event_type == "Celo" or SafeBoolContains(combined_norm, [r"\bcelo\b", r"se deja montar"]))
    insemination_flag = int(event_type == "Inseminación" or SafeBoolContains(combined_norm, [r"artificial insemination", r"\binsem"]))
    calving_flag = int(event_type == "Parto")
    dryoff_flag = int(event_type == "Secado")

    return pd.Series({
        "diagnosis_text": diagnosis_text,
        "medication_text": medication_text,
        "treatment_text": treatment_text,
        "location_text": location_text,
        "diagnosis_category": diagnosis_category,
        "treatment_category": treatment_category,
        "medication_category": medication_category,
        "body_condition_score": body_condition_score,
        "weight_kg": weight_kg,
        "liters_at_dryoff": liters_at_dryoff,
        "dcc_at_dryoff": dcc_at_dryoff,
        "del_at_dryoff": del_at_dryoff,
        "gestant_insem_number": gestant_insem_number,
        "from_group": from_group,
        "to_group": to_group,
        "semen_code": semen_code,
        "pregnancy_positive": pregnancy_positive,
        "pregnancy_negative": pregnancy_negative,
        "mastitis_flag": mastitis_flag,
        "digestive_flag": digestive_flag,
        "vaccine_flag": vaccine_flag,
        "reproductive_flag": reproductive_flag,
        "abortion_flag": abortion_flag,
        "antibiotic_flag": antibiotic_flag,
        "hormone_flag": hormone_flag,
        "dryoff_related_flag": dryoff_related_flag,
        "fresh_cow_flag": fresh_cow_flag,
        "rumen_bolus_flag": rumen_bolus_flag,
        "heat_flag": heat_flag,
        "insemination_flag": insemination_flag,
        "calving_flag": calving_flag,
        "dryoff_flag": dryoff_flag,
    })

if not events_df.empty:
    events_df = pd.concat([events_df, events_df.apply(EnrichEventRow, axis=1)], axis=1)

events_df.head(10)


,cow_id,page_number,lactation_number,event_type,event_type_clean,date_token,event_order,raw_event_text,anchor_year,year_inferred,...,abortion_flag,antibiotic_flag,hormone_flag,dryoff_related_flag,fresh_cow_flag,rumen_bolus_flag,heat_flag,insemination_flag,calving_flag,dryoff_flag
0,1204,1,1,Cambio tabla ali,Cambio_Tabla_Alim,22/08/2,0,User1 Condición corporal: 3.25 - DDUP:,NaN,2025,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1204,1,1,Condición corpo,Condicion_Corporal,22/08/2,1,"473 User1 Dry Off SE SECO CON 473 DEL, 20.75 L...",NaN,2025,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1204,1,1,Secado,Secado,22/08/2,2,User1 GESTANTE EN LA 7 INSEM. SE SECO PORQUE T...,NaN,2025,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0
3,1204,1,1,Cambio de grup,Cambio_Grupo,22/08/2,3,User1 Dns: VACUNA; Loc.: DD; Trat:,NaN,2025,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1204,1,1,Diagnósticos/Tr,Diagnostico_Tratamiento,22/08/2,4,starvacc; Med: STARVAC 1 d 1X2ml User1 Intramu...,NaN,2025,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,1204,1,1,Diagnósticos/Tr,Diagnostico_Tratamiento,22/08/2,5,ROTAVEC; Med: User1 Dns: Secado por baja; Loc....,NaN,2025,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
6,1204,1,1,Diagnósticos/Tr,Diagnostico_Tratamiento,22/08/2,6,Trat: CEFA SAFE; Med: CEFA SAFE 1 User1 d 1X4p...,NaN,2025,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,1204,1,1,Condición corpo,Condicion_Corporal,22/08/2,7,473 Admin + CARGADA DE 124,NaN,2025,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,1204,1,1,Control de Gest,Control_Gestacion,21/08/2,8,User1 + CARGADA DE 96,NaN,2025,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,1204,1,1,Control de Gest,Control_Gestacion,24/07/2,9,"User1 24/07/2025, Reconfirmacion 1 (90-",2025.0,2025,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [7]:
# Validaciones rápidas
if not events_df.empty:
    print("Rango de fechas:", events_df["date"].min(), "->", events_df["date"].max())
    print("Vacas únicas:", events_df["cow_id"].nunique())
    print("Tipos de evento:", events_df["event_type"].nunique())
    print("Tipos de evento limpios:", events_df["event_type_clean"].nunique())

    display(
        events_df[
            [
                "cow_id",
                "date",
                "event_type",
                "event_type_clean",
                "diagnosis_text",
                "medication_text",
                "treatment_text",
                "body_condition_score",
                "weight_kg",
                "liters_at_dryoff",
                "raw_event_text",
            ]
        ].head(30)
    )
else:
    print("No se extrajeron eventos.")

Rango de fechas: 2021-04-03 00:00:00 -> 2025-12-31 00:00:00
Vacas únicas: 67
Tipos de evento: 14
Tipos de evento limpios: 14


,cow_id,date,event_type,event_type_clean,diagnosis_text,medication_text,treatment_text,body_condition_score,weight_kg,liters_at_dryoff,raw_event_text
0,1204,2025-08-22,Cambio tabla ali,Cambio_Tabla_Alim,NaN,NaN,NaN,NaN,NaN,NaN,User1 Condición corporal: 3.25 - DDUP:
1,1204,2025-08-22,Condición corpo,Condicion_Corporal,NaN,NaN,NaN,NaN,NaN,NaN,"473 User1 Dry Off SE SECO CON 473 DEL, 20.75 L..."
2,1204,2025-08-22,Secado,Secado,NaN,NaN,NaN,NaN,NaN,NaN,User1 GESTANTE EN LA 7 INSEM. SE SECO PORQUE T...
3,1204,2025-08-22,Cambio de grup,Cambio_Grupo,VACUNA,NaN,NaN,NaN,NaN,NaN,User1 Dns: VACUNA; Loc.: DD; Trat:
4,1204,2025-08-22,Diagnósticos/Tr,Diagnostico_Tratamiento,VACUNA,STARVAC 1 d 1X2ml User1 Intramuscular,NaN,NaN,NaN,NaN,starvacc; Med: STARVAC 1 d 1X2ml User1 Intramu...
5,1204,2025-08-22,Diagnósticos/Tr,Diagnostico_Tratamiento,Secado por baja,NaN,NaN,NaN,NaN,NaN,ROTAVEC; Med: User1 Dns: Secado por baja; Loc....
6,1204,2025-08-22,Diagnósticos/Tr,Diagnostico_Tratamiento,NaN,CEFA SAFE 1 User1 d 1X4pcs Condición corporal:...,CEFA SAFE,NaN,NaN,NaN,Trat: CEFA SAFE; Med: CEFA SAFE 1 User1 d 1X4p...
7,1204,2025-08-22,Condición corpo,Condicion_Corporal,NaN,NaN,NaN,NaN,NaN,NaN,473 Admin + CARGADA DE 124
8,1204,2025-08-21,Control de Gest,Control_Gestacion,NaN,NaN,NaN,NaN,NaN,NaN,User1 + CARGADA DE 96
9,1204,2025-07-24,Control de Gest,Control_Gestacion,NaN,NaN,NaN,NaN,NaN,NaN,"User1 24/07/2025, Reconfirmacion 1 (90-"


In [8]:

# Guardar dataset a nivel evento: una fila por evento
event_columns = [
    "cow_id",
    "date",
    "event_type",
    "event_type_clean",
    "lactation_number",
    "event_order",
    "year_inferred",
    "diagnosis_text",
    "medication_text",
    "treatment_text",
    "location_text",
    "diagnosis_category",
    "treatment_category",
    "medication_category",
    "body_condition_score",
    "weight_kg",
    "liters_at_dryoff",
    "dcc_at_dryoff",
    "del_at_dryoff",
    "gestant_insem_number",
    "from_group",
    "to_group",
    "semen_code",
    "pregnancy_positive",
    "pregnancy_negative",
    "mastitis_flag",
    "digestive_flag",
    "vaccine_flag",
    "reproductive_flag",
    "abortion_flag",
    "antibiotic_flag",
    "hormone_flag",
    "dryoff_related_flag",
    "fresh_cow_flag",
    "rumen_bolus_flag",
    "heat_flag",
    "insemination_flag",
    "calving_flag",
    "dryoff_flag",
    "raw_event_text",
]

events_df = events_df[event_columns].copy()
events_df = events_df.sort_values(["cow_id", "date", "event_order"]).reset_index(drop=True)

OUTPUT_EVENTS.parent.mkdir(parents=True, exist_ok=True)
events_df.to_parquet(OUTPUT_EVENTS, index=False)

print("Guardado:", OUTPUT_EVENTS)
events_df.head()


Guardado: ../data/interim/events.parquet


,cow_id,date,event_type,event_type_clean,lactation_number,event_order,year_inferred,diagnosis_text,medication_text,treatment_text,...,antibiotic_flag,hormone_flag,dryoff_related_flag,fresh_cow_flag,rumen_bolus_flag,heat_flag,insemination_flag,calving_flag,dryoff_flag,raw_event_text
0,1204,2022-06-05,Entrada,Entrada,0,81,2022,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,User1
1,1204,2022-06-08,Cambio ID,Cambio_ID,0,80,2022,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,User1 Nacimiento
2,1204,2022-07-01,Cambio ID,Cambio_ID,0,79,2022,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,User1 2003 -> 99980
3,1204,2022-08-15,Diagnósticos/Tr,Diagnostico_Tratamiento,0,78,2022,NaN,DelproClien 99980 -> 1204,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,SHOT; Med: DelproClien 99980 -> 1204
4,1204,2022-09-27,Peso,Peso,0,76,2022,VACUNA,NaN,ONE,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,DelproClien Dns: VACUNA; Loc.: DD; Trat: ONE-


In [9]:

# Dataset diario: una fila por cow_id + date
# Aquí se conservan:
# - event_count
# - event_types concatenados
# - raw_event_text concatenado
# - one-hot encoding por tipo de evento
# - one-hot encoding clínico (diagnóstico / tratamiento / medicamento)
# - agregados clínicos / reproductivos / sanitarios

def JoinUniqueNonEmpty(series):
    vals = []
    for x in series.dropna().astype(str):
        x = x.strip()
        if x:
            vals.append(x)
    return " | ".join(sorted(set(vals)))

# base agregada
events_daily = (
    events_df
    .groupby(["cow_id", "date"], dropna=False)
    .agg(
        event_count=("event_type", "size"),
        event_types=("event_type", lambda s: " | ".join(sorted(set(s.astype(str))))),
        raw_event_text=("raw_event_text", lambda s: " || ".join(s.astype(str))),
        diagnosis_text=("diagnosis_text", JoinUniqueNonEmpty),
        medication_text=("medication_text", JoinUniqueNonEmpty),
        treatment_text=("treatment_text", JoinUniqueNonEmpty),
        diagnosis_categories=("diagnosis_category", JoinUniqueNonEmpty),
        treatment_categories=("treatment_category", JoinUniqueNonEmpty),
        medication_categories=("medication_category", JoinUniqueNonEmpty),
        body_condition_score=("body_condition_score", "max"),
        weight_kg=("weight_kg", "max"),
        liters_at_dryoff=("liters_at_dryoff", "max"),
        dcc_at_dryoff=("dcc_at_dryoff", "max"),
        del_at_dryoff=("del_at_dryoff", "max"),
        gestant_insem_number=("gestant_insem_number", "max"),
        pregnancy_positive=("pregnancy_positive", "max"),
        pregnancy_negative=("pregnancy_negative", "max"),
        mastitis_flag=("mastitis_flag", "max"),
        digestive_flag=("digestive_flag", "max"),
        vaccine_flag=("vaccine_flag", "max"),
        reproductive_flag=("reproductive_flag", "max"),
        abortion_flag=("abortion_flag", "max"),
        antibiotic_flag=("antibiotic_flag", "max"),
        hormone_flag=("hormone_flag", "max"),
        dryoff_related_flag=("dryoff_related_flag", "max"),
        fresh_cow_flag=("fresh_cow_flag", "max"),
        rumen_bolus_flag=("rumen_bolus_flag", "max"),
        heat_flag=("heat_flag", "max"),
        insemination_flag=("insemination_flag", "max"),
        calving_flag=("calving_flag", "max"),
        dryoff_flag=("dryoff_flag", "max"),
    )
    .reset_index()
)

# one-hot encoding a partir del tipo de evento
events_exploded = (
    events_df[["cow_id", "date", "event_type_clean"]]
    .dropna(subset=["event_type_clean"])
    .drop_duplicates()
)

event_ohe = pd.get_dummies(events_exploded["event_type_clean"], dtype=int)
event_ohe = pd.concat([events_exploded[["cow_id", "date"]], event_ohe], axis=1)
event_ohe = event_ohe.groupby(["cow_id", "date"], as_index=False).max()

# one-hot encoding clínico
diag_exploded = (
    events_df[["cow_id", "date", "diagnosis_category"]]
    .dropna(subset=["diagnosis_category"])
    .drop_duplicates()
)
diag_ohe = pd.get_dummies(diag_exploded["diagnosis_category"], dtype=int)
diag_ohe = pd.concat([diag_exploded[["cow_id", "date"]], diag_ohe], axis=1)
diag_ohe = diag_ohe.groupby(["cow_id", "date"], as_index=False).max()

trt_exploded = (
    events_df[["cow_id", "date", "treatment_category"]]
    .dropna(subset=["treatment_category"])
    .drop_duplicates()
)
trt_ohe = pd.get_dummies(trt_exploded["treatment_category"], dtype=int)
trt_ohe = pd.concat([trt_exploded[["cow_id", "date"]], trt_ohe], axis=1)
trt_ohe = trt_ohe.groupby(["cow_id", "date"], as_index=False).max()

med_exploded = (
    events_df[["cow_id", "date", "medication_category"]]
    .dropna(subset=["medication_category"])
    .drop_duplicates()
)
med_ohe = pd.get_dummies(med_exploded["medication_category"], dtype=int)
med_ohe = pd.concat([med_exploded[["cow_id", "date"]], med_ohe], axis=1)
med_ohe = med_ohe.groupby(["cow_id", "date"], as_index=False).max()

events_daily = events_daily.merge(event_ohe, on=["cow_id", "date"], how="left")
events_daily = events_daily.merge(diag_ohe, on=["cow_id", "date"], how="left")
events_daily = events_daily.merge(trt_ohe, on=["cow_id", "date"], how="left")
events_daily = events_daily.merge(med_ohe, on=["cow_id", "date"], how="left")

binary_cols = []
binary_cols += [c for c in EVENT_MAP.values() if c in events_daily.columns]
binary_cols += [c for c in events_daily.columns if c.startswith("diag_")]
binary_cols += [c for c in events_daily.columns if c.startswith("trt_")]
binary_cols += [c for c in events_daily.columns if c.startswith("med_")]
binary_cols += [
    "pregnancy_positive",
    "pregnancy_negative",
    "mastitis_flag",
    "digestive_flag",
    "vaccine_flag",
    "reproductive_flag",
    "abortion_flag",
    "antibiotic_flag",
    "hormone_flag",
    "dryoff_related_flag",
    "fresh_cow_flag",
    "rumen_bolus_flag",
    "heat_flag",
    "insemination_flag",
    "calving_flag",
    "dryoff_flag",
]
binary_cols = [c for c in binary_cols if c in events_daily.columns]

for c in binary_cols:
    events_daily[c] = events_daily[c].fillna(0).astype(int)

events_daily = events_daily.sort_values(["cow_id", "date"]).reset_index(drop=True)

OUTPUT_EVENTS_DAILY.parent.mkdir(parents=True, exist_ok=True)
events_daily.to_parquet(OUTPUT_EVENTS_DAILY, index=False)

print("Guardado:", OUTPUT_EVENTS_DAILY)
print("\nColumnas one-hot disponibles:")
print(sorted([c for c in events_daily.columns if c in EVENT_MAP.values()]))
print("\nColumnas clínicas OHE disponibles:")
print(sorted([c for c in events_daily.columns if c.startswith(("diag_", "trt_", "med_"))]))

events_daily.head(20)


Guardado: ../data/interim/events_daily.parquet

Columnas one-hot disponibles:
['Cambio_Grupo', 'Cambio_ID', 'Cambio_ID_Tran', 'Cambio_Tabla_Alim', 'Celo', 'Condicion_Corporal', 'Control_Gestacion', 'Diagnostico_Tratamiento', 'Entrada', 'Inseminacion', 'Invitacion_Visita', 'Parto', 'Peso', 'Secado']

Columnas clínicas OHE disponibles:
['diag_alta', 'diag_bolo_rumensin', 'diag_digestivo', 'diag_gnrh', 'diag_mastitis', 'diag_other', 'diag_pg', 'diag_presynch', 'diag_problema_podal', 'diag_revision', 'diag_secado', 'diag_secado_baja', 'diag_sincronizacion', 'diag_vacia', 'diag_vacuna', 'diag_vitamina', 'med_antibiotico', 'med_bolo_rumensin', 'med_cefa', 'med_cidr', 'med_gnrh', 'med_other', 'med_pg', 'med_vacuna', 'trt_antibiotico', 'trt_bolo_rumensin', 'trt_cefa', 'trt_cidr', 'trt_gnrh', 'trt_other', 'trt_pg', 'trt_vacuna', 'trt_vitamina']


,cow_id,date,event_count,event_types,raw_event_text,diagnosis_text,medication_text,treatment_text,diagnosis_categories,treatment_categories,...,trt_vacuna,trt_vitamina,med_antibiotico,med_bolo_rumensin,med_cefa,med_cidr,med_gnrh,med_other,med_pg,med_vacuna
0,1204,2022-06-05,1,Entrada,User1,,,,,,...,0,0,0,0,0,0,0,0,0,0
1,1204,2022-06-08,1,Cambio ID,User1 Nacimiento,,,,,,...,0,0,0,0,0,0,0,0,0,0
2,1204,2022-07-01,1,Cambio ID,User1 2003 -> 99980,,,,,,...,0,0,0,0,0,0,0,0,0,0
3,1204,2022-08-15,1,Diagnósticos/Tr,SHOT; Med: DelproClien 99980 -> 1204,,DelproClien 99980 -> 1204,,,,...,0,0,0,0,0,0,0,1,0,0
4,1204,2022-09-27,2,Diagnósticos/Tr | Peso,DelproClien Dns: VACUNA; Loc.: DD; Trat: ONE- ...,VACUNA,,ONE,diag_vacuna,trt_antibiotico,...,0,0,0,0,0,0,0,0,0,0
5,1204,2023-05-17,1,Diagnósticos/Tr,LEPTOFERM 5; Med: LEPTOFERM 5 User1 1 d 1X2ml ...,,LEPTOFERM 5 User1 1 d 1X2ml 64 kg ALTURA 92,,,,...,0,0,0,0,0,0,0,1,0,0
6,1204,2023-05-23,1,Peso,User1 Dns: LEPTOFERM 5; Loc.: DD; Trat:,LEPTOFERM 5,,,diag_other,,...,0,0,0,0,0,0,0,0,0,0
7,1204,2023-06-29,1,Peso,User1 300 kg Talla: 127.1,,,,,,...,0,0,0,0,0,0,0,0,0,0
8,1204,2023-07-12,1,Invitación Visita,407) DR RAFA ( 339 kg altura de 127,,,,,,...,0,0,0,0,0,0,0,0,0,0
9,1204,2023-07-13,2,Diagnósticos/Tr | Invitación Visita,X1; Med: LUTALYSE 1 d 1X5ml DelproClien Intram...,,LUTALYSE 1 d 1X5ml DelproClien Intramuscular P...,,,,...,0,0,0,0,0,0,0,0,1,0


In [10]:

# Auditoría rápida de categorías clínicas extraídas
if not events_df.empty:
    print("Diagnósticos más comunes:")
    display(
        events_df["diagnosis_text"]
        .dropna()
        .value_counts()
        .head(20)
        .rename_axis("diagnosis_text")
        .reset_index(name="count")
    )

    print("Medicamentos más comunes:")
    display(
        events_df["medication_text"]
        .dropna()
        .value_counts()
        .head(20)
        .rename_axis("medication_text")
        .reset_index(name="count")
    )

    print("Tratamientos más comunes:")
    display(
        events_df["treatment_text"]
        .dropna()
        .value_counts()
        .head(20)
        .rename_axis("treatment_text")
        .reset_index(name="count")
    )

    print("Categorías de diagnóstico más comunes:")
    display(
        events_df["diagnosis_category"]
        .dropna()
        .value_counts()
        .rename_axis("diagnosis_category")
        .reset_index(name="count")
    )

    print("Categorías de tratamiento más comunes:")
    display(
        events_df["treatment_category"]
        .dropna()
        .value_counts()
        .rename_axis("treatment_category")
        .reset_index(name="count")
    )

    print("Categorías de medicamento más comunes:")
    display(
        events_df["medication_category"]
        .dropna()
        .value_counts()
        .rename_axis("medication_category")
        .reset_index(name="count")
    )


Diagnósticos más comunes:


,diagnosis_text,count
0,VACUNA,299
1,Vacia,134
2,Mastitis clínica,95
3,Limpiando,79
4,BOLO RUMENSIN,73
5,VITAMINA,58
6,Protocolo sincronizacion,51
7,Presynch,44
8,ALTA,43
9,Secar,42


Medicamentos más comunes:


,medication_text,count
0,STARVAC 1 d 1X2ml User1 Intramuscular,50
1,BOLO User1 RUMENSIN 1 d 1X1pcs,44
2,STARVAC 1 d 1X2ml DelproClien Intramuscular,30
3,LUTALYSE 1 d 1X5ml DelproClien Intramuscular,16
4,User1 Vacas frescas,14
5,STARVAC 1 d 1X2ml User1 Intramuscular +,13
6,"DelproClien Vacas frescas, Available for Vet V...",13
7,"User1 Vacas frescas, Available for Vet Visit",12
8,BOLO DelproClien RUMENSIN 1 d 1X1pcs,11
9,"STARVAC 1 d 1X2ml User1 Intramuscular 106, BEC...",10


Tratamientos más comunes:


...

,diagnosis_category,count
0,diag_vacuna,299
1,diag_vacia,134
2,diag_other,126
3,diag_mastitis,119
4,diag_bolo_rumensin,73
5,diag_vitamina,58
6,diag_sincronizacion,53
7,diag_presynch,44
8,diag_alta,44
9,diag_secado,42


Categorías de tratamiento más comunes:


,treatment_category,count
0,trt_pg,164
1,trt_other,100
2,trt_cidr,78
3,trt_bolo_rumensin,76
4,trt_cefa,71
5,trt_antibiotico,49
6,trt_gnrh,38
7,trt_vacuna,10
8,trt_vitamina,3


Categorías de medicamento más comunes:


,medication_category,count
0,med_other,408
1,med_vacuna,182
2,med_bolo_rumensin,85
3,med_pg,77
4,med_cefa,77
5,med_antibiotico,66
6,med_cidr,4
7,med_gnrh,3
